In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader

from dataset_loaders import MPNNDataset

In [2]:
device = "cuda"

dataset = MPNNDataset(
    device,
    "graph_random_regular_graph_n9_d4",
    program="graph_coloring",
)

loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [3]:
class ToyMPNN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.lin_msg1 = nn.Linear(in_dim, hidden_dim)
        self.lin_msg2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, X, edge_index):
        """
        Forward pass for the MPNN with batching support.

        X: Tensor of shape (batch_size, num_nodes, in_dim)
        edge_index: Tensor of shape (2, num_edges) with the indices of the edges.
        batch_size: The number of graphs in the batch.
        """
        # Reshape X to have shape (batch_size * num_nodes, in_dim)
        batch_size = X.size(0)
        num_nodes = X.size(1)
        X_reshaped = X.view(-1, X.size(2))  # (batch_size * num_nodes, in_dim)

        # Get source and destination nodes from edge_index
        src, dst = edge_index
        num_edges = src.size(0)

        offsets = torch.arange(batch_size, device=X.device) * num_nodes  # (batch_size,)
        offsets = offsets.view(-1, 1).expand(-1, num_edges).reshape(-1)  # (batch_size * num_edges,)

        src = src.repeat(batch_size) + offsets
        dst = dst.repeat(batch_size) + offsets

        # Layer 1: Message Passing
        m1 = self.lin_msg1(X_reshaped[src])  # Message passing from src nodes
        agg1 = torch.zeros(
            X_reshaped.size(0), m1.size(1), device=X.device
        )  # Aggregation tensor
        agg1.index_add_(0, dst, m1)  # Aggregate messages at destination nodes
        h1 = F.relu(agg1)  # Apply ReLU activation

        # Layer 2: Message Passing
        m2 = self.lin_msg2(
            h1[src]
        )  # Message passing from src nodes (after first aggregation)
        agg2 = torch.zeros(
            h1.size(0), m2.size(1), device=X.device
        )  # Aggregation tensor
        agg2.index_add_(0, dst, m2)  # Aggregate messages at destination nodes
        h2 = F.relu(agg2)  # Apply ReLU activation

        return h2

In [4]:
# Hyperparameters
in_dim = 1
hidden_dim = 4
out_dim = in_dim  # number of classes
lr = 0.01
epochs = 100

# Model, loss, optimizer
model = ToyMPNN(in_dim, hidden_dim, out_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0001)

In [5]:
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    total_loss = 0

    for batch in loader:
        X = batch[0]
        y = batch[1].reshape(-1, batch[1].size(2))
        out = model(X, dataset.edge_index)  # forward pass
        # print("out", out, "y", y)
        loss = criterion(out, y)  # compute loss
        loss.backward()  # backward pass
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(name, param.grad.abs().mean())
        optimizer.step()  # update weights
        total_loss += loss.item()

    if (epoch + 1) % 5 == 0:
        # pred = out.argmax(dim=1)
        # acc = (pred == y).float().mean()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 5, Loss: 0.2337
Epoch 10, Loss: 0.1957
Epoch 15, Loss: 0.1700
Epoch 20, Loss: 0.1725
Epoch 25, Loss: 0.1752
Epoch 30, Loss: 0.1706
Epoch 35, Loss: 0.1683
Epoch 40, Loss: 0.1689
Epoch 45, Loss: 0.1692
Epoch 50, Loss: 0.1689
Epoch 55, Loss: 0.1685
Epoch 60, Loss: 0.1684
Epoch 65, Loss: 0.1683
Epoch 70, Loss: 0.1683
Epoch 75, Loss: 0.1683
Epoch 80, Loss: 0.1683
Epoch 85, Loss: 0.1683
Epoch 90, Loss: 0.1683
Epoch 95, Loss: 0.1683
Epoch 100, Loss: 0.1683
